In [1]:
import polars as pl

In [2]:
pl.Config.set_tbl_cols(-1)  # Display all columns
pl.Config.set_tbl_rows(-1)  # Display all rows

polars.config.Config

In [3]:
#script to create a metada file for wagtail run
#inside the metadata file, we will have the following columns:
#1. sample_id
#2. total input reads (from qc reports)
#3. total retained reads (from qc reports)
#4. total retained reads percentage (#3/#2*100)
#5-18. deblur stats result
#19. total reads for mapping (from samtools flagstat reports line 1)
#20. total all mapped reads with percentage (from samtools flagstat reports line 7)
#21. total all primary mapped reads with percentage (from samtools flagstat reports line 8)

#1-18 is combining from the qc-stats report with deblur stats result
#19-21 is from samtools flagstat report and take only line 1, 7, and 8

In [5]:
#read the qc-stats report and deblur stats result
qc_stats = pl.read_csv("qc-stats.csv")
deblur_stats = pl.read_csv("deblur-stats.csv")


In [11]:
qc_stats[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases
str,i64,i64,i64,i64,i64
"""V1V2""",73176,73031,104,104,41


In [12]:
#add percentage of total retained reads to the qc_stats dataframe
#calculate the total retained reads/total input reads * 100 and save it to a new column called retained-reads-percentage
qc_stats1 = qc_stats.with_columns((pl.col("total-retained-reads") / pl.col("total-input-reads") * 100).alias("retained-reads-percentage"))
qc_stats1[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage
str,i64,i64,i64,i64,i64,f64
"""V1V2""",73176,73031,104,104,41,99.801848


In [7]:
deblur_stats[:4]

sample-id,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""V1V2""",73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0


In [13]:
#combine the qc_stats1 and deblur_stats dataframe
#we will use the sample-id column to join the two dataframes
metadata = qc_stats1.join(deblur_stats, on="sample-id")
metadata[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0


In [53]:
#read the samtools flagstat report
samtools_flagstat = pl.read_csv("V1V2_flagstat_tsv.txt", separator= '\t', has_header=False)
samtools_flagstat[:10]
index = [0, 1, 2, 3, 7, 8, 9]

In [54]:
samtools_flagstat1 = samtools_flagstat.filter(pl.Series(range(len(samtools_flagstat))).is_in(index))
print(samtools_flagstat1)

shape: (7, 3)
┌──────────┬──────────┬───────────────────────────────────┐
│ column_1 ┆ column_2 ┆ column_3                          │
│ ---      ┆ ---      ┆ ---                               │
│ str      ┆ str      ┆ str                               │
╞══════════╪══════════╪═══════════════════════════════════╡
│ 164      ┆ 0        ┆ total (QC-passed reads + QC-fail… │
│ 143      ┆ 0        ┆ primary                           │
│ 0        ┆ 0        ┆ secondary                         │
│ 21       ┆ 0        ┆ supplementary                     │
│ 100.00%  ┆ N/A      ┆ mapped %                          │
│ 143      ┆ 0        ┆ primary mapped                    │
│ 100.00%  ┆ N/A      ┆ primary mapped %                  │
└──────────┴──────────┴───────────────────────────────────┘


In [55]:
#transpose the samtools_flagstat dataframe and use the column_3 content as the column name
samtools_flagstat2 = samtools_flagstat1.select([pl.col("column_1"), pl.col("column_2")]).transpose()
samtools_flagstat2.columns = samtools_flagstat1["column_3"].to_list()
samtools_flagstat2[:4]

total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,str,str,str,str,str,str
"""164""","""143""","""0""","""21""","""100.00%""","""143""","""100.00%"""
"""0""","""0""","""0""","""0""","""N/A""","""0""","""N/A"""


In [57]:
#delete row 2 of samtools_flagstat1
row_to_delete = [1]
samtools_flagstat3 = samtools_flagstat2.filter(~pl.Series(range(len(samtools_flagstat2))).is_in(row_to_delete))
samtools_flagstat3[:4]

total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,str,str,str,str,str,str
"""164""","""143""","""0""","""21""","""100.00%""","""143""","""100.00%"""


In [58]:
#add one more column with the column name as sample-id and the value is the same as the sample-id in the metadata dataframe
samtools_flagstat4 = samtools_flagstat3.with_columns(metadata["sample-id"])
samtools_flagstat4[:4]

total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %,sample-id
str,str,str,str,str,str,str,str
"""164""","""143""","""0""","""21""","""100.00%""","""143""","""100.00%""","""V1V2"""


In [59]:
#combine the metadata dataframe with the samtools_flagstat4 dataframe based on the sample-id column
metadata1 = metadata.join(samtools_flagstat4, on="sample-id")
metadata1[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference,total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,"""164""","""143""","""0""","""21""","""100.00%""","""143""","""100.00%"""


In [60]:
#read some file and then append the metadata1 dataframe to the file
metadata1.write_csv("metadata.tsv", separator="\t", has_header=True)

In [63]:
#testing to combine 2 metadata files into a single metadata file without duplicating all headers
metadata3 = pl.read_csv("metadata.tsv", separator="\t", has_header=True)
metadata2 = pl.read_csv("metadata2.tsv", separator="\t", has_header=True)
metadata2[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference,total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str
"""V1V3""",73178,63031,104,104,41,89.801848,63031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""


In [64]:
combined_df = metadata3.vstack(metadata2)
combined_df[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference,total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""
"""V1V3""",73178,63031,104,104,41,89.801848,63031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""


In [65]:
metadata4 = pl.read_csv("metadata3.tsv", separator="\t", has_header=True)
metadata5 = pl.read_csv("metadata4.tsv", separator="\t", has_header=True)

In [67]:
combined_df2 = combined_df.vstack(metadata4).vstack(metadata5)
combined_df2[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference,total (QC-passed reads + QC-failed reads),primary,secondary,supplementary,mapped %,primary mapped,primary mapped %
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""
"""V1V3""",73178,63031,104,104,41,89.801848,63031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""
"""V1V4""",73178,63031,104,104,41,89.801848,63031,2992,51264,194,13683,0,0,31,607,143,13075,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""
"""V1V6""",73178,63031,104,104,41,89.801848,64031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,164,143,0,21,"""100.00%""",143,"""100.00%"""


In [68]:
#read some file and then append the metadata1 dataframe to the file
combined_df2.write_csv("combined_metadata.tsv", separator="\t", has_header=True)